In [ ]:
from azureml.core import Workspace
from azureml.core import Environment
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.compute_target import ComputeTargetException

ws = Workspace.from_config()
test_docker_env = Environment("onprem-disc-connection-docker-env") # change the environment name accordingly

# Choose a name for your cluster.
compute_instance =  {
        "compute-name": "test-git",
        "compute-size": "STANDARD_DS3_V2",
        "startup-script": "./copy_config.sh",
    }

try:
    compute_target = ComputeTarget(workspace=ws, name=compute_instance["compute-name"])
    print('Found existing compute target.')
except ComputeTargetException:
    print('Creating a new compute target...')
    compute_config = AmlCompute.provisioning_configuration(vm_size='STANDARD_NC6',
                                                           max_nodes=4)

    # Create the cluster.
    compute_target = ComputeTarget.create(ws, compute_instance["compute-name"], compute_config)

    compute_target.wait_for_completion(show_output=True)

# Use get_status() to get a detailed status for the current AmlCompute.
print(compute_target.get_status().serialize())

### Pull docker image for disc connection from SoFa artifactory. For this you need to supply username and password for authentication

In [ ]:
# change the following base image to the one you would like to pull
test_docker_env.docker.base_image = "artifactory.sofa.dev/virtual-lab-docker-local/disc-connection-azureml"
test_docker_env.python.user_managed_dependencies = True # True: You are responsible for installing all necessary Python libraries, typically in your docker image.
# Set the container registry information.
test_docker_env.docker.base_image_registry.address = "artifactory.sofa.dev"
test_docker_env.docker.base_image_registry.username = "your ecb email address"
test_docker_env.docker.base_image_registry.password = "your SoFa artifactory identity token"

### Submit a run to AzureML 
- Copy credential and key chenages over to the docker container
- Run `python check_disc_connection.py` to test the basic connection to a public data lab in DISC
- Following section of code will create a new Jobs in Azure ML 

In [ ]:
from azureml.core import Experiment, ScriptRunConfig
from azureml.core.runconfig import DockerConfiguration

docker_config = DockerConfiguration(use_docker=True)

# the "disc-connection-azureml" docker image doesn't contain any credentials, keys, tokens. You need to copy for example your keytab file, 
# odbc.ini which contains devo username, password and other important configuration, files that you have applied changes and important for
# connecting DISC/DEVO
command = "bash copy_config.sh && python check_disc_connection.py".split()
config = ScriptRunConfig(
                source_directory=".",
                command=command,
                compute_target=compute_target,
                environment= test_docker_env,
                docker_runtime_config=docker_config  #comment this line, if you are not using the dockerfile approach
            )

run = Experiment(ws,'onprem-disc-connection-docker-env').submit(config)

## Register environment, after this step the custom environment is visible in Environments on AzureML

In [ ]:
# Run following line to register your custom env to the Environments
registered_env = test_docker_env.register(ws)